<a href="https://colab.research.google.com/github/SithuminiNimthara/Research_Project/blob/main/shoreline_yolov11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics opencv-python-headless pandas matplotlib tqdm pyyaml --quiet

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/Research"

# Dataset
ZIP_PATH = f"{BASE_DIR}/datasets/shoreline_seg_dataset.zip"
EXTRACT_DIR = f"{BASE_DIR}/shoreline_data/extracted"
TEST_FRAMES_DIR = f"{BASE_DIR}/shoreline_data/test_frames"

# YOLO11 folders
VERSION = "yolo11"
RUN_NAME = "shoreline_seg_yolo11_v1"

RUNS_DIR = f"{BASE_DIR}/runs/{VERSION}"
MODELS_DIR = f"{BASE_DIR}/models/{VERSION}"
EVAL_DIR = f"{BASE_DIR}/evaluations/{VERSION}"

# Nest points
NEST_CSV = f"{BASE_DIR}/nest_points/nest_points.csv"

# Create folders if missing
os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print("✅ Paths ready")
print("ZIP_PATH      :", ZIP_PATH)
print("EXTRACT_DIR   :", EXTRACT_DIR)
print("RUNS_DIR      :", RUNS_DIR)
print("MODELS_DIR    :", MODELS_DIR)
print("EVAL_DIR      :", EVAL_DIR)
print("NEST_CSV      :", NEST_CSV)

In [ ]:
!unzip -qo "{ZIP_PATH}" -d "{EXTRACT_DIR}"
!find "{EXTRACT_DIR}" -maxdepth 5 -name "data.yaml" -print

In [ ]:
data_yaml = f"{EXTRACT_DIR}/data.yaml"

if not os.path.exists(data_yaml):
    raise FileNotFoundError(f"❌ data.yaml not found: {data_yaml}")

print("✅ Found data.yaml:", data_yaml)

Train YOLO11 segmentation model

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")   # nano version for faster training

In [ ]:
model.train(
    data=data_yaml,
    epochs=80,
    imgsz=640,
    batch=16,
    project=RUNS_DIR,
    name=RUN_NAME,
    exist_ok=True
)

In [ ]:
import os
import shutil

run_dir = f"{RUNS_DIR}/{RUN_NAME}"
best_pt = f"{run_dir}/weights/best.pt"
save_pt = f"{MODELS_DIR}/shoreline_seg_v11_best.pt"

if not os.path.exists(best_pt):
    raise FileNotFoundError(f"❌ best.pt not found: {best_pt}")

shutil.copy(best_pt, save_pt)

print("✅ Best YOLO11 model saved to:", save_pt)
print("✅ Training run folder:", run_dir)

Display training result images

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{run_dir}/results.png"))
display(Image(filename=f"{run_dir}/val_batch0_pred.jpg"))

Read training metrics

In [ ]:
import pandas as pd

csv_path = f"{run_dir}/results.csv"
df11 = pd.read_csv(csv_path)

print("✅ Results file loaded")
print(df11.columns.tolist())

df11.head()

Plot YOLO11 metrics

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

if "metrics/mAP50(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/mAP50(B)"], label="mAP@50(B)")
if "metrics/mAP50-95(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/mAP50-95(B)"], label="mAP@50-95(B)")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("YOLO11 Training mAP")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

if "metrics/precision(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/precision(B)"], label="Precision")
if "metrics/recall(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/recall(B)"], label="Recall")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("YOLO11 Precision and Recall")
plt.legend()
plt.grid(True)
plt.show()

Save metric plots into evaluations/yolo11/

In [ ]:
plt.figure(figsize=(8,5))

if "metrics/mAP50(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/mAP50(B)"], label="mAP@50(B)")
if "metrics/mAP50-95(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/mAP50-95(B)"], label="mAP@50-95(B)")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("YOLO11 Training mAP")
plt.legend()
plt.grid(True)

map_plot_path = f"{EVAL_DIR}/yolo11_map_curve.png"
plt.savefig(map_plot_path, bbox_inches="tight")
plt.show()

print("✅ Saved:", map_plot_path)

In [ ]:
plt.figure(figsize=(8,5))

if "metrics/precision(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/precision(B)"], label="Precision")
if "metrics/recall(B)" in df11.columns:
    plt.plot(df11["epoch"], df11["metrics/recall(B)"], label="Recall")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("YOLO11 Precision and Recall")
plt.legend()
plt.grid(True)

pr_plot_path = f"{EVAL_DIR}/yolo11_precision_recall_curve.png"
plt.savefig(pr_plot_path, bbox_inches="tight")
plt.show()

print("✅ Saved:", pr_plot_path)